# Latent Mesh Diffusion Computer — Phase 1 Training

**Before running:** Get a HF token at `huggingface.co/settings/tokens` (Write scope).

This notebook:
1. Clones the repo
2. Installs dependencies
3. Converts HF datasets → binary shards
4. Trains Phase 1 (250M parameters)
5. Pushes checkpoints to your HF Hub model repo

In [ ]:
# @title 1. Mount Google Drive (optional, for checkpoint backup)
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/latent_mesh'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted at {DRIVE_DIR}')

In [ ]:
# @title 2. Clone repo & install dependencies
import os, sys, subprocess
WORK = '/content'
os.chdir(WORK)

REPO = 'https://github.com/mohamed-raad/latent-mesh-diffusion'
if not os.path.isdir('latent-mesh-diffusion'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
os.chdir('latent-mesh-diffusion')
sys.path.insert(0, 'NoProp/src')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch>=2.5', 'transformers', 'datasets', 'numpy',
    'accelerate', 'huggingface-hub', 'tensorboard', 'bitsandbytes'], check=True)

print('Repo cloned, deps installed')

In [ ]:
# @title 3. Login to HuggingFace Hub
from huggingface_hub import login, HfApi
import getpass

HF_TOKEN = getpass.getpass('Enter your HF token:')
HF_REPO_ID = 'mohamed99raad/Latent-Mesh-Model'  # @param {type:"string"}

login(token=HF_TOKEN, add_to_git_credential=True)

api = HfApi()
api.create_repo(HF_REPO_ID, repo_type='model', private=True, exist_ok=True)
print(f'Logged in as {api.whoami()["name"][0]}')
print(f'Model repo: {HF_REPO_ID}')

In [ ]:
# @title 4. Convert HF datasets → binary shards
import time, os
from bin_converter import convert_hf_to_bin
from train_pipeline import PHASE_CONFIGS

cfg = PHASE_CONFIGS['phase1_250m']
BIN_DIR = '/content/bin_data'
os.makedirs(BIN_DIR, exist_ok=True)

for ds_spec in cfg.datasets:
    hf_path = ds_spec['hf_path']
    hf_config = ds_spec.get('hf_config')
    tag = hf_path.replace('/', '_') + (f'_{hf_config}' if hf_config else '')
    bin_path = os.path.join(BIN_DIR, tag)
    if os.path.isfile(os.path.join(bin_path, 'index.json')):
        print(f'  Already cached: {tag}')
        continue
    t0 = time.time()
    print(f'  Converting {tag}...')
    convert_hf_to_bin(hf_path, bin_path, hf_config=hf_config,
                      max_docs=200000, shard_size=10000,
                      max_seq_len=cfg.canvas_len)
    print(f'  Done ({time.time()-t0:.0f}s)')

print(f'All data ready in {BIN_DIR}')

In [ ]:
# @title 5. Train Phase 1
import time, os, shutil
from train_pipeline import TrainingOrchestrator

LOG_DIR = '/content/logs'
CKPT_DIR = '/content/checkpoints'

orchestrator = TrainingOrchestrator(
    log_dir=LOG_DIR,
    checkpoint_base=CKPT_DIR,
    use_packing=False,
)

print('Starting Phase 1 (250M) — 50,000 steps')
t0 = time.time()
orchestrator.run_phase(cfg)
elapsed = time.time() - t0
print(f'Training complete in {elapsed/3600:.1f}h')

# Backup to Drive
if os.path.isdir('/content/drive/MyDrive'):
    shutil.copytree(CKPT_DIR, f'{DRIVE_DIR}/checkpoints', dirs_exist_ok=True)
    print('Backed up to Google Drive')

In [ ]:
# @title 6. Push checkpoints to HuggingFace Hub
from huggingface_hub import upload_folder

for item in os.listdir(CKPT_DIR):
    item_path = os.path.join(CKPT_DIR, item)
    if os.path.isdir(item_path):
        print(f'Uploading {item}...')
        upload_folder(
            folder_path=item_path,
            repo_id=HF_REPO_ID,
            repo_type='model',
            path_in_repo=item,
            token=HF_TOKEN,
        )
        print(f'  Done')

upload_folder(
    folder_path=LOG_DIR,
    repo_id=HF_REPO_ID,
    repo_type='model',
    path_in_repo='logs',
    token=HF_TOKEN,
)
print(f'All uploaded to https://huggingface.co/{HF_REPO_ID}')

In [ ]:
# @title 7. (Optional) Resume from previous checkpoint
# If you need to resume, pull from HF Hub:
# from huggingface_hub import snapshot_download
# snapshot_download(repo_id=HF_REPO_ID, local_dir=CKPT_DIR, token=HF_TOKEN)
# print(f'Checkpoints restored from {HF_REPO_ID}')
print('To resume next time: uncomment the cell above')